# L12b Advanced: Learning a Policy with Q-Learning
In the L12b lecture, the traders in our market follow a Wolfram rule, which is a policy table: for each neighbor sum, the table gives the trader's next intention. The lecture hands each trader its table and changes one entry. In this notebook, we ask where such a table could come from, and study Q-learning, a method with which an agent learns a policy from experience instead of being handed one. This material is optional and is not needed for the lecture or its example.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
> * __Understand the Q-function and its update rule:__ Explain what the state-action-value function $Q(s,a)$ estimates, and how one experience updates the estimate through the reward and a one-step lookahead.
> * __Understand the Q-learning algorithm:__ Describe how an agent balances exploring random actions with choosing greedy ones, and how the learned $Q(s,a)$ gives a policy $\pi(s)$.
> * __Understand when Q-learning converges:__ State the learning-rate and exploration conditions under which the estimates converge to the optimal policy.

A Wolfram rule is a policy table, and Q-learning is one way an agent could learn such a table. The traders in the lecture are handed theirs.

Let's get started!
___

<div>
    <center>
        <img src="figs/Fig-Q-Schematic.svg" width="580"/>
    </center>
</div>

## Q-Learning
In Markov decision processes (MDPs), the __state-action-value function__ $Q(s,a)$ (also called the Q-function) represents the expected cumulative discounted reward of taking action $a$ in state $s$ and following the optimal policy thereafter. We have a model of the environment defined by the transition probabilities $T\left(s^{\prime}\,|\,s,a\right)$ and the reward function $R(s,a)$. 

However, in many real-world scenarios, we do not have access to this model. Instead, we can learn the optimal policy directly from experience using __Q-learning__, a model-free reinforcement learning algorithm.

Q-learning iteratively estimates the state action-value function $Q(s, a)$ by conducting repeated experiments $t=1,2,\ldots$ in the world $\mathcal{W}$. 
In each experiment, an agent in state $s\in\mathcal{S}$ takes action $a\in\mathcal{A}$, receives a reward $r$, and (potentially) transitions to a new state $s^{\prime}$. After each experiment $t$, the agent updates its estimate of $Q(s, a)$ using the update rule:
$$
\begin{equation*}
Q_{t+1}(s,a)\leftarrow{\underbrace{Q_{t}(s,a)}_{\text{old value}}}+\alpha_{t}\cdot\underbrace{\left(r+\gamma\cdot\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime}) - Q_{t}(s,a)\right)}_{\text{new value}}\quad{t = 1,2,3,\ldots}
\end{equation*}
$$
where $0<\alpha_{t} <{1}$ is the learning rate parameter at time $t$, and $0<\gamma<{1}$ is the discount factor. 
We estimate the policy function $\pi:\mathcal{S}\rightarrow\mathcal{A}$ by selecting the action $a$ that maximizes $Q(s,a)$ at each state $s$:
$$
\begin{equation*}
\pi(s) = \arg\max_{a\in\mathcal{A}}Q(s,a)
\end{equation*}
$$

#### Algorithm
Initialize $Q(s,a)$ arbitrarily for all $s\in\mathcal{S}$, and $a\in\mathcal{A}$.
Set the hyperparameters: learning rate $\alpha_{t}$, the discount factor $\gamma$, the exploration rate $\epsilon_{t}$, the maximum number of iterations $\texttt{maxiter}$, and the convergence tolerance $\delta$. 

For $s\in\mathcal{S}$
1. Initialize the trial counter $t\gets{1}$ and set $\texttt{converged}\gets\texttt{false}$
2. While $\texttt{converged} $ is $\texttt{false}$ __do__:
    1. Roll a random number $p\in[0,1]$. Compute $\epsilon_{t}={t^{-1/3}}\cdot\left(K\cdot\log(t)\right)^{1/3}$ where $K=|\mathcal{A}|$ is the number of actions.
    2. If $p\leq\epsilon_{t}$, choose a random (uniform) action $a_{t}\in\mathcal{A}$. Otherwise, choose a greedy action $a_{t} = \text{arg}\max_{a\in\mathcal{A}}{Q_{t}(s,a)}$.
    3. Take action $a_{t}$, observe the reward $r$ from the __world__ and transition to the next state $s^{\prime}$.
    4. Update the state-action-value function: $Q_{t+1}(s,a_{t})\leftarrow{Q_{t}(s,a_{t})}+\alpha_{t}(s,a_{t})\cdot\underbrace{\left(r+\gamma\cdot\overbrace{\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime})}^{\text{one-step lookahead}} - Q_{t}(s,a_{t})\right)}_{\text{new information}}$.
    5. Update the state $s\leftarrow{s^{\prime}}$, the learning rate of the pair updated in step 4 (see Convergence below), and the counter $t\leftarrow{t+1}$
    6. Convergence check: If the $Q(s,a)$ has bounded change $\lVert{Q_{t+1}(s,a) - Q_{t}(s,a)}\rVert\leq\delta$, then the algorithm has converged. Set $\texttt{converged}\gets\texttt{true}$.
    7. Otherwise: if $t\geq\texttt{maxiter}$, then set $\texttt{converged}\gets\texttt{true}$ and notify the caller that the maximum iteration limit was reached without convergence. Proceed to next state.
    8. Otherwise: continue to the next iteration.
3. End While
4. End For

#### Convergence
Q-learning converges to the optimal policy under two key theoretical conditions (assuming the Markov property holds for the environment):
* __Learning rate decay__: The learning rate $\alpha_{t}$ must satisfy $\sum_{t=0}^\infty \alpha_t(s, a) = \infty$ and $\sum_{t=0}^\infty \alpha_t^2(s, a) < \infty$ for all state-action pairs, ensuring sufficient initial updates while stabilizing over time.

  A common choice gives each pair its own rate, $\alpha_{0}/(1+n)$, where $0<\alpha_{0}<1$ and $n$ is the number of earlier updates of that pair. Both conditions then hold for every pair that is visited infinitely often.
* __Sufficient exploration__: Tabular convergence results require every relevant state--action pair to be visited infinitely often together with appropriate learning-rate and stationarity assumptions. Merely having $\epsilon_t>0$ at every finite time is not sufficient; the exploration schedule and state visitation process must provide infinite coverage (for example, GLIE-style conditions).

Q-learning produces a policy $\pi(s) = \arg\max_{a}Q(s,a)$ that maps states to actions. In the lecture, we use pre-computed policies encoded as Wolfram rules to simulate market dynamics. However, in more advanced agent-based models, agents can learn their policies using Q-learning as described above.
___

## Summary
In this notebook, we looked at Q-learning as one way an agent could learn a policy table, instead of being handed one as the traders in the L12b lecture are.

> __Key Takeaways:__
>
> * __The Q-function and its update rule:__ We defined the state-action-value function $Q(s,a)$ as the expected cumulative discounted reward of taking action $a$ in state $s$ and acting optimally afterward, and wrote the update that moves each estimate toward the reward plus the discounted best value of the next state. This let us learn from experience when the transition probabilities and rewards of the world are not known.
> * __The algorithm balances exploring and exploiting:__ We laid out the algorithm step by step, with an exploration rate that shrinks over time, so that an agent sometimes tries a random action and otherwise takes the greedy one. This showed how the learned $Q(s,a)$ gives the policy $\pi(s)=\arg\max_{a}Q(s,a)$, a table of the same kind as a Wolfram rule.
> * __Convergence needs a decaying learning rate and enough exploration:__ We stated the two conditions under which Q-learning converges to the optimal policy: a learning rate that sums to infinity while its squares sum to a finite value, and visits to every relevant state-action pair infinitely often. This let us say when we can trust a learned policy table.

Q-learning gives us a way to ask how the policy tables of an agent-based market could be learned instead of assigned.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___